In [1]:
!pip install transformers torch --quiet


In [10]:
import torch
import torch.nn as nn
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from transformers.cache_utils import DynamicCache
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


Device: cuda


In [11]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

base_model = GPT2LMHeadModel.from_pretrained("gpt2")

# freezing GPT-2 w
for param in base_model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in base_model.parameters())
print(f"GPT-2 parameters (all frozen): {total_params:,}")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT-2 parameters (all frozen): 124,439,808


In [12]:
class PrefixEncoder(nn.Module):

    def __init__(self, prefix_length, num_layers, hidden_size, prefix_hidden_size=512):
        super().__init__()

        self.prefix_length = prefix_length
        self.num_layers = num_layers
        self.hidden_size = hidden_size

        # Embedding prefix tokens
        self.embedding = nn.Embedding(prefix_length, prefix_hidden_size)

        # output size = num_layers * 2 * hidden_size  (كل layer تحتاج K و V)
        self.mlp = nn.Sequential(
            nn.Linear(prefix_hidden_size, prefix_hidden_size),
            nn.Tanh(),
            nn.Linear(prefix_hidden_size, num_layers * 2 * hidden_size)
        )

    def forward(self, batch_size, device):
        # [prefix_length]
        prefix_tokens = torch.arange(self.prefix_length, device=device)

        # [prefix_length, prefix_hidden_size]
        embeds = self.embedding(prefix_tokens)

        # [prefix_length, num_layers * 2 * hidden_size]
        past_key_values = self.mlp(embeds)

        past_key_values = past_key_values.view(
            self.prefix_length, self.num_layers * 2, self.hidden_size
        )

        # [num_layers * 2, prefix_length, hidden_size]
        past_key_values = past_key_values.permute(1, 0, 2)

        #  batch: [num_layers * 2, batch_size, prefix_length, hidden_size]
        past_key_values = past_key_values.unsqueeze(1).expand(-1, batch_size, -1, -1)

        #  head dimension (GPT-2 small: 12 heads, head_dim=64)
        num_heads = 12
        head_dim = self.hidden_size // num_heads

        past_key_values = past_key_values.view(
            self.num_layers * 2, batch_size, self.prefix_length, num_heads, head_dim
        ).permute(0, 1, 3, 2, 4)

        #  tuple (key, value)
        cache = DynamicCache()
        for i in range(self.num_layers):
            key   = past_key_values[2 * i]
            value = past_key_values[2 * i + 1]
            cache.update(key, value, layer_idx=i)
        return cache



In [19]:
class PrefixTuningModel(nn.Module):

    def __init__(self, base_model, prefix_length=10):
        super().__init__()
        self.base_model = base_model
        config = base_model.config

        self.prefix_encoder = PrefixEncoder(
            prefix_length=prefix_length,
            num_layers=config.n_layer,
            hidden_size=config.n_embd,
        )
        self.prefix_length = prefix_length

    def forward(self, input_ids, attention_mask=None, labels=None):
        batch_size = input_ids.size(0)
        device     = input_ids.device

        past_key_values = self.prefix_encoder(batch_size, device)

        if attention_mask is not None:
            prefix_mask    = torch.ones(batch_size, self.prefix_length, device=device)
            attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)

        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            labels=labels,
        )
        return outputs

    def generate_text(self, prompt, max_new_tokens=50):
        self.eval()
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            prefix_tokens = torch.zeros(
                1, self.prefix_length, dtype=torch.long, device=device
            ).fill_(tokenizer.eos_token_id)

            input_ids      = torch.cat([prefix_tokens, inputs["input_ids"]], dim=1)
            attention_mask = torch.ones_like(input_ids)

            output = self.base_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.8,
                pad_token_id=tokenizer.eos_token_id,
            )

        output = output[:, self.prefix_length:]
        return tokenizer.decode(output[0], skip_special_tokens=True)



## Comparing num of Parameters

In [20]:
model = PrefixTuningModel(base_model, prefix_length=10)
model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

print(f"Trainable parameters (prefix only) : {trainable:,}")
print(f"Total parameters                   : {total:,}")
print(f"Trainable %                        : {100 * trainable / total:.2f}%")
print()


Trainable parameters (prefix only) : 9,723,392
Total parameters                   : 134,163,200
Trainable %                        : 7.25%



In [21]:
class SimpleTextDataset(Dataset):

    def __init__(self, texts, tokenizer, max_length=64):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt"
        )

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        input_ids      = self.encodings["input_ids"][idx]
        attention_mask = self.encodings["attention_mask"][idx]
        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         input_ids.clone()
        }


sample_texts = [
    "The weather today is sunny and warm.",
    "Machine learning is a subfield of artificial intelligence.",
    "Python is a popular programming language for data science.",
    "Deep learning models require large amounts of training data.",
    "Natural language processing helps computers understand text.",
    "Transfer learning allows models to reuse pretrained knowledge.",
    "The cat sat on the mat and looked out the window.",
    "Scientists discovered a new species of deep-sea fish.",
]

dataset    = SimpleTextDataset(sample_texts, tokenizer)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

print(f"Dataset size : {len(dataset)} samples")
print(f"Batches      : {len(dataloader)}")


Dataset size : 8 samples
Batches      : 4


In [22]:
def train(model, dataloader, optimizer, num_epochs=5):
    model.train()

    for epoch in range(num_epochs):
        total_loss = 0

        for batch in dataloader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            optimizer.zero_grad()

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")


# Optimizer - only for prefix parameters
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

print("Train Started!")
train(model, dataloader, optimizer, num_epochs=5)


Train Started!
Epoch 1/5 | Loss: 4.9451
Epoch 2/5 | Loss: 1.3913
Epoch 3/5 | Loss: 1.3219
Epoch 4/5 | Loss: 1.3042
Epoch 5/5 | Loss: 1.0259


In [23]:
prompts = [
    "Machine learning",
    "Deep learning",
    "Natural language",
]

print("Text Generation\n")
for prompt in prompts:
    generated = model.generate_text(prompt, max_new_tokens=30)
    print(f"Prompt   : {prompt}")
    print(f"Generated: {generated}")
    print("-" * 60)


Text Generation

Prompt   : Machine learning
Generated: Machine learning is rapidly becoming the most powerful tool in the world — but it's still a bit hard to build a neural system that can solve most of the problems
------------------------------------------------------------
Prompt   : Deep learning
Generated: Deep learning is a new approach to artificial intelligence that is rapidly becoming a real-world problem in science and technology. However, one very big challenge is that there
------------------------------------------------------------
Prompt   : Natural language
Generated: Natural language and language skills are not as common as you might think.

This is something to consider, given the number of people currently studying English, though
------------------------------------------------------------


In [24]:
import os

torch.save(model.prefix_encoder.state_dict(), "prefix_weights.pt")

size_mb = os.path.getsize("prefix_weights.pt") / (1024 * 1024)
print(f"prefix_weights.pt")
print(f"File size: {size_mb:.2f} MB")
print()
print("Comparing:")
print(f"  GPT-2   ≈ 500 MB")
print(f"  prefix  ≈ {size_mb:.1f} MB !")


prefix_weights.pt
File size: 37.09 MB

Comparing:
  GPT-2   ≈ 500 MB
  prefix  ≈ 37.1 MB !
